#  Análisis de Inversión y TCO: Flota Mixta (44t)

Este notebook presenta la evaluación financiera del caso de negocio para la transición a vehículos pesados eléctricos. Comparamos la operativa 100% Diésel vs Estrategia Mixta, evaluando **Compra, Leasing y Renting**.

##  Metodología y Fundamentos Financieros (TCO)

El análisis TCO (Total Cost of Ownership) busca calcular el coste **real y acumulado** de la flota a 5 años, yendo más allá del precio de compra:

$$TCO = CF_0 + \sum_{t=1}^{n} \frac{CF_t}{(1 + r)^t}$$

### 1. Coste Operativo ($OpEx_t$) frente a la Inversión ($CapEx$)
*   **CapEx (Inversión Inicial):** Desembolso de capital en el Año 0 ($CF_0$). Para el caso eléctrico, incluye el camión y los cargadores rápidos.
*   **OpEx (Coste Operativo):** Consumo de energía, mantenimientos ($M\&S$) y seguros, ajustados anualmente por la inflación ($i$):
$$OpEx_t = (Energía_t + M\&S_t) \cdot (1+i)^t$$

### 2. El "Descuento" de Hacienda: El Escudo Fiscal ($\tau = 25\%$)
La metodología evalúa los flujos **después de impuestos**, permitiendo deducir los gastos estructurales según la modalidad:

*   **Compra Directa:** Genera ahorro vía amortizaciones ($\frac{CapEx}{n}$).
    $$CF_t = -OpEx_t + (OpEx_t + \frac{CapEx}{n}) \cdot \tau$$
*   **Leasing Financiero:** Permite deducir la cuota íntegra ($Cuota_t$).
    $$CF_t = -OpEx_t - Cuota_t + (OpEx_t + Cuota_t) \cdot \tau$$
*   **Renting (Full-Service):** La cuota incluye $M\&S$, por lo que solo la energía es variable.
    $$CF_t = -Energía_t - Cuota_t + (Energía_t + Cuota_t) \cdot \tau$$

> **Nota Técnica sobre Renting:** En esta modalidad se utiliza el término **Energía** en lugar de **OpEx** porque el contrato de Renting (*Full-Service*) ya incorpora los costes de mantenimiento y seguro en la cuota fija. El único gasto operativo directo que asume la empresa fuera de la cuota es el combustible o la electricidad.

### 3. Parámetros Críticos y Rentabilidad
*   **VAN:** El modelo utiliza el **WACC ($r$)** para traer los flujos futuros al presente, determinando el **Punto de Equilibrio (Break-even)**.
*   **Incentivos:** El modelo integra la subvención del Plan MOVES III como flujo positivo en el Año 1.

---
*Resumen: El modelo no solo suma lo que la empresa paga, sino que resta estratégicamente el ahorro en impuestos, determinando la eficiencia financiera real de la transición.*



In [36]:
import sys
import os
import pandas as pd
import plotly.graph_objects as go

# Asegurar acceso al núcleo del optimizador
sys.path.append(os.path.abspath(os.getcwd()))

from logistic_core.utils.strategic_analyzer import StrategicAnalyzer

print("✅ Motor Estratégico Cargado")

✅ Motor Estratégico Cargado


# 1. WACC E INFLACIÓN CONSTANTE


## 1. Ejecución del Modelo TCO
Simulamos el escenario de inversión para la flota mixta.

In [37]:
# =================================================================
# PANEL DE CONTROL GLOBAL: PARÁMETROS DE INVERSIÓN 
# =================================================================

# --- 1. MACROECONOMÍA Y ESCENARIO ---
HORIZONTE_AÑOS = 5           # Duración del análisis financiero
KMS_ANUALES_POR_CAMION = 130_000 

# Nota: WACC e INFLACION ahora aceptan escalares o LISTAS de 5 valores
WACC = 0.07                   # Tasa de descuento (VAN). Ej: [0.07, 0.07, 0.08, 0.08, 0.09]
INFLACION = 0.02              # Crecimiento anual de costes. Ej: [0.02, 0.03, 0.05, 0.02, 0.02]
TAX_RATE = 0.25               

# --- 2. TECNOLOGÍA DIÉSEL (EURO VI) ---
D_CAPEX = 140_000             
D_CONSUMO_100KM = 33.0        
D_PRECIO_FUEL = 1.45          
D_MANT_ANUAL = 9_000          # Mantenimiento + Neumáticos
D_SEGURO_ANUAL = 3_500        
D_RESIDUAL_PCT = 0.30         # Valor de reventa (30%)
D_RENTING_MES = 2_800         
D_LEASING_MES = 2_600         

# --- 3. TECNOLOGÍA ELÉCTRICA (BEV 44T) ---
EV_CAPEX = 350_000            
EV_CHARGER = 50_000           
EV_MOVES = 90_000             
EV_CONSUMO_KM = 1.30          
EV_PRECIO_KWH = 0.18          
EV_MANT_ANUAL = 4_000         
EV_SEGURO_ANUAL = 4_500       
EV_RESIDUAL_PCT = 0.25        
EV_RENTING_MES = 5_600        
EV_LEASING_MES = 5_200        

# --- 4. COMPOSICIÓN DE LA FLOTA PARA EL TEST ---
N_DIESEL = 33
N_EV = 18

# =================================================================
# INYECCIÓN TOTAL AL MOTOR ESTRATÉGICO
# =================================================================

analyzer = StrategicAnalyzer(
    kms_anuales=KMS_ANUALES_POR_CAMION,
    wacc=WACC,
    inflación_anual=INFLACION,
    diesel_params={
        "capex": D_CAPEX,
        "consumo_l_100km": D_CONSUMO_100KM,
        "coste_combustible_l": D_PRECIO_FUEL,
        "mantenimiento_anual": D_MANT_ANUAL,
        "seguro_anual": D_SEGURO_ANUAL,
        "residual_pct": D_RESIDUAL_PCT
    },
    ev_params={
        "capex_truck": EV_CAPEX,
        "capex_charger": EV_CHARGER,
        "ayuda_moves": EV_MOVES,
        "consumo_kwh_km": EV_CONSUMO_KM,
        "coste_kwh": EV_PRECIO_KWH,
        "mantenimiento_anual": EV_MANT_ANUAL,
        "seguro_anual": EV_SEGURO_ANUAL,
        "residual_pct": EV_RESIDUAL_PCT
    },
    financiacion={
        "diesel_renting": D_RENTING_MES,
        "diesel_leasing": D_LEASING_MES,
        "ev_renting": EV_RENTING_MES,
        "ev_leasing": EV_LEASING_MES
    }
)

# 1. Ejecutamos el motor actualizado
resultados = analyzer.generar_tabla_comparativa(n_diesel=N_DIESEL, n_ev=N_EV)

# 2. DataFrame con los resultados
data_mixta = []
for modalidad, datos in resultados['flota_mixta'].items():
    data_mixta.append({
        "Modalidad": modalidad.upper(),
        "Nº Diésel": datos['n_diesel'],
        "Nº Eléctricos": datos['n_ev'],
        "Valor Residual Final": datos['valor_residual_total'],
        "TCO Total (VAN)": abs(datos['tco_total']),
        "Coste Medio (€/km)": datos['coste_km_medio']
    })

df_tco = pd.DataFrame(data_mixta)

# 3. Resumen de premisas de la inversión (incluyendo WACC e Inflación)
print(f"--- RESUMEN DE LA INVERSIÓN ---")
print(f"Horizonte de Análisis: {HORIZONTE_AÑOS} años")

# Detección dinámica de tipo (Escalar o Lista)
if isinstance(WACC, list):
    print(f"WACC (Serie): [{', '.join([f'{v:.1%}' for v in WACC])}]")
else:
    print(f"WACC: {WACC:.1%}")

if isinstance(INFLACION, list):
    print(f"Inflación (Serie): [{', '.join([f'{v:.1%}' for v in INFLACION])}]")
else:
    print(f"Inflación: {INFLACION:.1%}")

print(f"Valor Unitario Diésel: {D_CAPEX:_} €".replace('_', '.'))
print(f"Valor Unitario Eléctrico: {EV_CAPEX:_} €".replace('_', '.'))
print(f"Subvención MOVES por Eléctrico: {EV_MOVES:_} €".replace('_', '.'))
print("-" * 31)
print(f"Total Flota a gestionar: {N_DIESEL + N_EV} unidades")


# 4. Estilo profesional para la presentación
df_tco.style.format({
    "Valor Residual Final": "{:,.0f} €",
    "TCO Total (VAN)": "{:,.0f} €",
    "Coste Medio (€/km)": "{:.3f} €/km"
}).background_gradient(subset=["TCO Total (VAN)"], cmap="RdYlGn_r")



--- RESUMEN DE LA INVERSIÓN ---
Horizonte de Análisis: 5 años
WACC: 7.0%
Inflación: 2.0%
Valor Unitario Diésel: 140.000 €
Valor Unitario Eléctrico: 350.000 €
Subvención MOVES por Eléctrico: 90.000 €
-------------------------------
Total Flota a gestionar: 51 unidades


,Modalidad,Nº Diésel,Nº Eléctricos,Valor Residual Final,TCO Total (VAN),Coste Medio (€/km)
0,COMPRA,33,18,"3,186,000 €","16,591,537 €",0.500 €/km
1,LEASING,33,18,"3,186,000 €","16,086,506 €",0.485 €/km
2,RENTING,33,18,0 €,"15,700,650 €",0.474 €/km


### 2. Análisis Visual de Costes Acumulados

In [38]:
modos = ["compra", "leasing", "renting"]
nombres = ["Compra Directa", "Leasing Financiero", "Renting Operativo"]

y_diesel = [abs(resultados['flota_mixta'][m]['tco_diesel_subtotal']) for m in modos]
y_ev = [abs(resultados['flota_mixta'][m]['tco_ev_subtotal']) for m in modos]

fig = go.Figure(data=[
    go.Bar(name='TCO Diésel', x=nombres, y=y_diesel, marker_color='#475569'),
    go.Bar(name='TCO Eléctrico', x=nombres, y=y_ev, marker_color='#10b981')
])

fig.update_layout(
    barmode='stack', 
    title='TCO a 5 Años: Comparativa de Modalidades de Adquisición',
    yaxis_title='Euros (€)',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)

fig.show()

# Llamada directa al nuevo método de la clase
fig_evolu = analyzer.plot_van_evolution(n_diesel=N_DIESEL, n_ev=N_EV)
fig_evolu.show()

# 2. WACC E INFLACIÓN VARIABLES


In [39]:
# =================================================================
# ESCENARIO DINÁMICO: ANÁLISIS DE SENSIBILIDAD (SERIES)
# =================================================================

# Definimos las series temporales para no sobrescribir los escalares previos
WACC_SERIE = [0.07, 0.075, 0.08, 0.085, 0.09]      
INFLACION_SERIE = [0.02, 0.03, 0.05, 0.03, 0.02]   

# Creamos una instancia específica para el análisis de variables
analyzer_var = StrategicAnalyzer(
    kms_anuales=KMS_ANUALES_POR_CAMION,
    wacc=WACC_SERIE,
    inflación_anual=INFLACION_SERIE,
    diesel_params={
        "capex": D_CAPEX,
        "consumo_l_100km": D_CONSUMO_100KM,
        "coste_combustible_l": D_PRECIO_FUEL,
        "mantenimiento_anual": D_MANT_ANUAL,
        "seguro_anual": D_SEGURO_ANUAL,
        "residual_pct": D_RESIDUAL_PCT
    },
    ev_params={
        "capex_truck": EV_CAPEX,
        "capex_charger": EV_CHARGER,
        "ayuda_moves": EV_MOVES,
        "consumo_kwh_km": EV_CONSUMO_KM,
        "coste_kwh": EV_PRECIO_KWH,
        "mantenimiento_anual": EV_MANT_ANUAL,
        "seguro_anual": EV_SEGURO_ANUAL,
        "residual_pct": EV_RESIDUAL_PCT
    },
    financiacion={
        "diesel_renting": D_RENTING_MES,
        "diesel_leasing": D_LEASING_MES,
        "ev_renting": EV_RENTING_MES,
        "ev_leasing": EV_LEASING_MES
    }
)

# 1. Ejecución del motor dinámico
resultados_var = analyzer_var.generar_tabla_comparativa(n_diesel=N_DIESEL, n_ev=N_EV)

# 2. Construcción del DataFrame de sensibilidad
data_sensibilidad = []
for modalidad, datos in resultados_var['flota_mixta'].items():
    data_sensibilidad.append({
        "Modalidad": modalidad.upper(),
        "Nº Diésel": datos['n_diesel'],
        "Nº Eléctricos": datos['n_ev'],
        "Valor Residual Final": datos['valor_residual_total'],
        "TCO Total (VAN)": abs(datos['tco_total']),
        "Coste Medio (€/km)": datos['coste_km_medio']
    })

df_tco_var = pd.DataFrame(data_sensibilidad)

# 3. Resumen detallado de la evolución de tasas
print(f"--- RESUMEN DE LA INVERSIÓN (TCO DINÁMICO) ---")
print(f"Horizonte de Análisis: {HORIZONTE_AÑOS} años")

wacc_str = ", ".join([f"{v:.1%}" for v in WACC_SERIE])
print(f"WACC (Serie por año): [{wacc_str}]")

inf_str = ", ".join([f"{v:.1%}" for v in INFLACION_SERIE])
print(f"Inflación (Serie por año): [{inf_str}]")

print(f"Valor Unitario Diésel: {D_CAPEX:_} €".replace('_', '.'))
print(f"Valor Unitario Eléctrico: {EV_CAPEX:_} €".replace('_', '.'))
print("-" * 37)
print(f"Total Flota a gestionar: {N_DIESEL + N_EV} unidades")


# 4. Presentación mediante Styler
df_tco_var.style.format({
    "Valor Residual Final": "{:,.0f} €",
    "TCO Total (VAN)": "{:,.0f} €",
    "Coste Medio (€/km)": "{:.3f} €/km"
}).background_gradient(subset=["TCO Total (VAN)"], cmap="RdYlGn_r")

--- RESUMEN DE LA INVERSIÓN (TCO DINÁMICO) ---
Horizonte de Análisis: 5 años
WACC (Serie por año): [7.0%, 7.5%, 8.0%, 8.5%, 9.0%]
Inflación (Serie por año): [2.0%, 3.0%, 5.0%, 3.0%, 2.0%]
Valor Unitario Diésel: 140.000 €
Valor Unitario Eléctrico: 350.000 €
-------------------------------------
Total Flota a gestionar: 51 unidades


,Modalidad,Nº Diésel,Nº Eléctricos,Valor Residual Final,TCO Total (VAN),Coste Medio (€/km)
0,COMPRA,33,18,"3,186,000 €","16,817,859 €",0.507 €/km
1,LEASING,33,18,"3,186,000 €","16,058,317 €",0.484 €/km
2,RENTING,33,18,0 €,"15,670,498 €",0.473 €/km


### 2. Análisis Visual de Costes Acumulados

In [40]:
# =================================================================
# VISUALIZACIÓN: ESCENARIO DINÁMICO (SENSIBILIDAD)
# =================================================================

modos = ["compra", "leasing", "renting"]
nombres = ["Compra Directa", "Leasing Financiero", "Renting Operativo"]

# Utilizamos los resultados variables (resultados_var)
y_diesel_var = [abs(resultados_var['flota_mixta'][m]['tco_diesel_subtotal']) for m in modos]
y_ev_var = [abs(resultados_var['flota_mixta'][m]['tco_ev_subtotal']) for m in modos]

fig_tco_var = go.Figure(data=[
    go.Bar(name='TCO Diésel', x=nombres, y=y_diesel_var, marker_color='#475569'),
    go.Bar(name='TCO Eléctrico', x=nombres, y=y_ev_var, marker_color='#10b981')
])

fig_tco_var.update_layout(
    barmode='stack', 
    title='TCO Dinámico (Series Variables): Comparativa de Modalidades',
    yaxis_title='Euros (€)',
    template='plotly_white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5)
)

fig_tco_var.show()

# Gráfico de evolución del VAN utilizando el analizador dinámico
# Este gráfico reflejará ahora el impacto de la inflación y WACC variables en el tiempo
fig_evolu_var = analyzer_var.plot_van_evolution(n_diesel=N_DIESEL, n_ev=N_EV)
fig_evolu_var.show()
